# Sistema de Detección Vial Integrado

Esta notebook integra y sintetiza el trabajo de inteligencia artificial desarrollado a lo largo del proyecto. Su objetivo es presentar, en forma de hilo conductor, la toma de decisiones técnicas que dieron forma al sistema: desde la definición del problema y la elección del modelo, pasando por la implementación del pipeline completo, hasta la evaluación de los resultados obtenidos y su integración en la infraestructura de producción.

Las notebooks `04_1` y `04_2` profundizan en los experimentos de entrenamiento y la construcción del dataset, respectivamente. Esta notebook los referencia donde corresponde y actúa como punto de entrada para quien quiera entender el sistema de forma integral.

## 1. Definición del Enfoque Analítico

En este apartado se describe la tarea abordada, el flujo de datos que atraviesa el componente de IA y la justificación técnica de las decisiones de diseño adoptadas.

### 1.1 Tarea Abordada y Formato de los Datos

El sistema resuelve un problema de **detección de objetos** y **seguimiento (tracking)** sobre fotogramas extraídos de videos capturados desde vehículos en movimiento. Las tres clases objetivo son:

- **D40 (Bache/Pozo):** Deterioros puntuales que representan riesgo inmediato para la seguridad vial.
- **D20 (Piel de Cocodrilo):** Grietas interconectadas que indican desgaste progresivo del pavimento.
- **Calle de Tierra:** Tramos no pavimentados, de relevancia para la planificación de obras de asfaltado.

El ciclo del dato en el componente de IA opera de la siguiente manera:

- **Entrada del sistema:** Videos en formato `.mp4` o `.webm` registrados por cámaras a bordo. Estos se suben a través de la API y quedan almacenados en MinIO. El **worker de preprocesamiento** descarga cada video, extrae 1 de cada 6 frames y aplica dos filtros antes de enviar el frame al siguiente paso: uno de **movimiento** (descarta frames donde el vehículo está detenido, comparando la diferencia entre frames consecutivos) y uno de **calidad** (detecta imágenes borrosas o con brillo insuficiente).

- **Datos de entrenamiento:** Imágenes `.jpg` anotadas en el formato estándar de YOLO (archivos `.txt` con coordenadas normalizadas de bounding boxes). El dataset combina capturas locales de Moreno y alrededores con imágenes del dataset global *RDD2022*, curadas y balanceadas (ver [Notebook 04_2](04_2_creacion_dataset_mixto_crudo.ipynb)).

- **Datos de sincronización GPS:** Junto a cada video, el sistema espera un archivo `.json` asociado que contiene una lista de puntos GPS con su marca temporal (`elapsed_ms`). El worker de inferencia utiliza este JSON para asignar coordenadas geográficas reales a cada detección, interpolando el punto GPS más cercano al momento en que apareció el daño en el video.

- **Salida del sistema:** Cada detección válida se persiste en la base de datos PostGIS como un registro georreferenciado, incluyendo el tipo de daño, el nivel de confianza del modelo y la ruta al frame anotado almacenado en MinIO.

### 1.2 Justificación de la Arquitectura (YOLO26m)

La familia de modelos **YOLO** (You Only Look Once) fue seleccionada desde el inicio del proyecto como la arquitectura base para la detección. La elección se apoyó en dos razones concretas:

- **Integración nativa con el ecosistema Ultralytics.** El stack completo del proyecto —entrenamiento, evaluación, inferencia y tracking— se ejecuta dentro del framework de Ultralytics. Esto permitió unificar los flujos de trabajo: el mismo objeto `YOLO` que se usó para entrenar en Google Colab es el que corre en producción (`best.pt`) dentro del worker de inferencia. No era necesario integrar una librería externa ni adaptar formatos de salida.

- **Escalabilidad controlada entre variantes.** Uno de los aprendizajes clave del proceso experimental fue que la variante de YOLO importa más que la versión. Empezamos con YOLO26 Small y llegamos a la versión **Medium (`yolo26m.pt`)** como resultado directo de los experimentos documentados en la [Notebook 04_1](04_1_documentacion_entrenamiento.ipynb). La versión Small no lograba distinguir texturas asfálticas complejas: generaba falsos positivos ante sombras de árboles o parches de humedad, situaciones frecuentes en las calles de Moreno. La versión Medium, más profunda en su extracción de características, redujo ese problema de forma notable, aunque no de manera completa.

Dentro de esa decisión, dos configuraciones específicas resultaron determinantes:

- **Resolución de 1024px (vs. el estándar de 640px):** Los baches pequeños o ubicados cerca del horizonte visual de la cámara pierden demasiados píxeles a 640px, especialmente al combinarse con el mosaico de YOLO. Al entrenar a 1024px, el modelo recibe suficiente información visual para detectarlos antes de que el vehículo pase sobre ellos.

- **Hiperparámetros adaptados al dominio vial:** Se desactivaron los giros verticales (`flipud=0.0`) porque la calzada siempre mantiene la misma orientación relativa respecto a la cámara. Se redujo la intensidad del mosaico (`mosaic=0.5`) para evitar que los baches pequeños se vuelvan indetectables al combinarse con otras imágenes. Y se aumentó la tolerancia a cambios de brillo y saturación (`hsv_s/v=0.5`) para simular variaciones de luz a lo largo del día.

> **Arquitecturas en evaluación:** Como paso siguiente, se está evaluando **RT-DETR** (Real-Time Detection Transformer), una arquitectura basada en Vision Transformers que, en lugar de predecir a partir de una cuadrícula fija, aplica mecanismos de atención global sobre toda la imagen. Esto podría mejorar la sensibilidad en daños finos como las grietas D20. Los resultados de esa evaluación se incorporarán en entregas futuras.

### 1.3 Del Experimento a la Decisión Final

La configuración actual no fue una decisión de diseño tomada a priori: es el resultado de tres iteraciones de entrenamiento, cada una corrigiendo los problemas identificados en la anterior. El camino completo —incluyendo los errores, los diagnósticos y las métricas de cada intento— está documentado en la [Notebook 04_1](04_1_documentacion_entrenamiento.ipynb).

En síntesis, los problemas que guiaron el rediseño fueron: exceso de imágenes de fondo en el primer intento (que enseñó al modelo que no había nada que detectar), sobreajuste en el segundo (generado por la combinación de data augmentation estático de Roboflow con el dinámico de YOLO), y resolución insuficiente para capturar baches pequeños. La decisión de usar imágenes crudas, inyectar solo muestras útiles del dataset global y escalar a Medium + 1024px responde directamente a cada uno de esos hallazgos.

El dataset sobre el que se entrenó el modelo final también tiene su propio proceso de construcción, documentado en la [Notebook 04_2](04_2_creacion_dataset_mixto_crudo.ipynb).

## 2. Implementación de la Línea de Base Funcional

En esta sección se detalla el funcionamiento lógico del pipeline que transforma el video crudo registrado en el municipio en registros geoespaciales estructurados y reportes ejecutivos enriquecidos, describiendo las herramientas y la lógica implementada en los componentes del sistema.

### 2.1 Pipeline de Procesamiento Distribuido (Workers)

El procesamiento de datos multimedia es demandante a nivel de cómputo. Para evitar congestionar la API principal de cara al usuario, el procesamiento se divide en dos etapas lógicas asíncronas orquestadas mediante colas de mensajes en **Redis**:

1. **Worker de Preprocesamiento (`worker_preprocesamiento.py`):** Su objetivo es limpiar la entrada y reducir el volumen de datos redundantes antes de que lleguen a la red neuronal.
   - **Descarga:** Extrae el video del bucket `videos-crudos` en MinIO.
   - **Corrección de orientación:** Si el video fue capturado en vertical (alto > ancho), rota los fotogramas automáticamente 90° en sentido antihorario para asegurar la consistencia del horizonte.
   - **Downsampling temporal:** Extrae únicamente 1 de cada 6 fotogramas (reduciendo los FPS a un volumen manejable sin perder resolución de detalles).
   - **Filtro de movimiento:** Mediante la diferencia absoluta entre fotogramas consecutivos, evalúa el porcentaje de cambio píxel a píxel. Si la variación es inferior al 2% (indicando que el vehículo de inspección se encuentra detenido en un semáforo o intersección), el frame se descarta de inmediato para no saturar la base de datos de detecciones duplicadas.
   - **Subida temporal:** Los frames válidos filtrados se suben de forma numerada al bucket temporal `frames-procesados` y se encola el ID en Redis (`cola_inferencia`).

2. **Worker de Inferencia y Tracking (`worker.py`):** Consume la cola de inferencia para detectar y persistir los desperfectos.
   - **Inferencia y Seguimiento:** Carga el modelo optimizado `best.pt` y procesa los fotogramas ordenados cronológicamente. Utiliza la implementación de **ByteTrack** para asignar identificadores de seguimiento únicos a las anomalías en movimiento.
   - **Filtro de Horizonte (50%):** A fin de evitar detecciones falsas en elementos externos (cielo, árboles, cables de luz u otros obstáculos), el worker descarta cualquier caja cuyo centroide vertical se ubique en la mitad superior del fotograma (`y_centro < alto_imagen * 0.50`), garantizando que la red neuronal solo analice la superficie de la calzada.
   - **Persistencia Física e Imágenes Anotadas:** Cuando detecta un desperfecto nuevo o actualiza uno existente con mayor confianza, dibuja la caja delimitadora, guarda el frame modificado en el bucket final `detecciones` de MinIO y almacena los metadatos y la geometría en PostGIS.

### 2.2 Integración Geoespacial y Sincronización Telemetría-Video

La conversión del análisis visual en un registro geográfico real requiere de dos lógicas críticas:

* **Sincronización Telemetría-Video:** El archivo `.json` de metadatos del video contiene un array de coordenadas registradas por el dispositivo móvil. Dado que la tasa de muestreo del GPS y los fotogramas del video no coinciden exactamente, el worker calcula la diferencia absoluta temporal entre el frame procesado (`tiempo_ms` extraído del nombre del archivo) y la marca de telemetría (`elapsed_ms`). Mediante una función de búsqueda mínima, interpola la latitud y longitud correspondientes al momento exacto de la captura.

* **Duplicación Espacial Inteligente (PostGIS):** Un bache puede persistir a lo largo de varios metros en el video. Para consolidar detecciones idénticas sin generar registros duplicados en la base de datos, el worker implementa un mecanismo híbrido:
  1. **Duplicación Visual:** Si el tracking de ByteTrack mantiene un `track_id` conocido que ya existe en nuestra base de datos para ese video, el worker no duplica el registro; simplemente actualiza la confianza y el frame asociado en MinIO si la predicción actual es más precisa.
  2. **Duplicación Geográfica:** Si el tracking visual se corta o se pierde temporalmente, se realiza una consulta espacial sobre la base de datos. Mediante la función espacial `ST_DWithin`, se verifica si ya existe una detección del mismo tipo dentro de un radio de tolerancia dinámico adaptado al tamaño promedio del daño:
     - **3 metros** para baches/pozos (`D40`).
     - **10 metros** para piel de cocodrilo/grietas (`D20`).
     - **30 metros** para el inicio o transcurso de una calle de tierra (`calle_tierra`).
     Si hay coincidencia geográfica, se asocia el nuevo track al registro existente para mitigar la intermitencia del modelo.

### 2.3 Enriquecimiento Urbano y Reportes de Inspección (OpenStreetMap y Ollama)

Una vez persistidas las anomalías, el sistema no se va a limitar a mostrar coordenadas en un mapa. En el router de reportes (`reporte.py`), se ejecuta un flujo de enriquecimiento y análisis semántico:

* **Contexto Geográfico (OpenStreetMap):** A través del módulo `geo_service.py`, las coordenadas de las detecciones se agrupan espacialmente (usando `ST_ClusterDBSCAN` para agrupar puntos a menos de 5 metros). Luego, se consulta la API geográfica para obtener el nombre de la calle, su jerarquía (Avenida, Ruta, Residencial) y la presencia de Puntos de Interés (POIs) clave a menos de 50 metros, como escuelas, hospitales o centros de salud.

* **Cálculo del Score de Prioridad Técnica Interno:** El sistema asigna un puntaje de severidad de forma automática:
  - Se otorgan puntos según la cantidad de daños en el tramo.
  - Se suman 3 puntos si es una calle de tierra (para priorizar obras de pavimentación).
  - Se agregan 5 puntos de peso si la anomalía se encuentra próxima a escuelas u hospitales.
  - Se multiplica el puntaje final por un factor de **1.5** si el tramo corresponde a una Avenida o Ruta, debido al flujo constante y la velocidad del tránsito.
  Este score es clave para que el LLM Local pueda decidir que calle tiene una urgencia real de ser reparada.

* **Generación de Reportes con LLM Local:** Con toda la información recopilada y formateada como contexto structured, la API realiza una solicitud HTTP a la instancia local de **Ollama** (`ollama:11434`) con un prompt especializado. El modelo (configurado con una temperatura baja de `0.1` para evitar alucinaciones) genera un reporte ejecutivo formal estructurado. Este reporte se persiste en la base de datos vinculándolo a los videos correspondientes.

* **Endpoint de Q&A de Inspección:** Adicionalmente, el endpoint `/api/v1/video/{video_id}/preguntar` permite al personal municipal realizar preguntas abiertas sobre una inspección (ej. *"¿Cuántos baches se detectaron cerca de la escuela?"* o *"¿Cuál es la confianza promedio?"*), respondiendo de forma precisa con el contexto de las detecciones guardadas y utilizando la información generada por los reportes.

### 2.4 Validación e Indicadores de Salud de la Infraestructura

Para garantizar que el sistema integrado esté operando realmente, la API provee endpoints específicos de monitoreo técnico:

- **Semáforo de Salud (`/api/v1/health`):** Realiza un ping en tiempo real a los 4 pilares tecnológicos del backend (PostgreSQL, Redis, MinIO y Ollama), devolviendo un estado unificado (`VERDE`, `AMARILLO` o `ROJO`).
- **Inventario del Sistema (`/api/v1/sistema/inventario`):** Cruza el volumen de la base de datos (videos e IDs registrados) con el almacenamiento físico de MinIO, listando la cantidad de archivos y tamaños en los buckets `videos-crudos`, `frames-procesados` y `detecciones`.

## 3. Evaluación Técnica Inicial

### 3.1 Resumen del Modelo y Resultados

El desarrollo y evolución de la arquitectura del modelo de visión computacional se encuentra documentado en la **[Notebook 04_1: Documentación y Bitácora de Entrenamiento](04_1_documentacion_entrenamiento.ipynb)**, la cual describe las iteraciones desde las versiones iniciales (*YOLO26 Small*) hasta consolidar la variante **YOLO26 Medium (`yolo26m.pt`)** entrenada a una resolución de **1024px** con el *Dataset Mixto V2 Crudo*.

Los indicadores generales del modelo actual son:
- **mAP50 Global:** 0.719 (71.9%)
- **Precisión Global:** 0.780 (78.0%)

Este desempeño se caracteriza por una alta precisión en la detección de grietas (`D20` - 74.6%), lo que reduce significativamente las falsas alarmas en el sistema, y una postura más conservadora en la detección de baches (`D40`), priorizando evitar falsos positivos ante la presencia de sombras o parches de humedad. Las métricas de validación completas y los análisis comparativos se consolidarán en un informe técnico de resultados independiente (`resultados.md`).

### 3.2 Limitaciones Actuales

A partir de las pruebas operativas en los videos de inspección del municipio de Moreno, se identificaron tres limitaciones principales en el modelo actual:

1. **Sensibilidad a factores ambientales:** Ambigüedad visual entre baches reales (`D40`) y sombras de árboles o parches de asfalto mojado
2. **Dependencia de iluminación y clima:** Disminución en la confianza de detección en condiciones climáticas adversas (como lluvia severa) o en inspecciones nocturnas con iluminación deficiente.
3. **Inflexibilidad del filtrado espacial:** El filtro de horizonte al 50% (`y_centro >= 0.50`) es estático. Movimientos bruscos del vehículo de inspección o pendientes empinadas en calles de tierra pueden causar que desperfectos reales queden fuera del área útil de procesamiento.

### 3.3 Oportunidades de Mejora y Futuras Evaluaciones

Para optimizar la capacidad de detección del sistema, se contemplan las siguientes líneas de trabajo:

1. **Aprendizaje Continuo y HITL:** Implementar un flujo *Human-in-the-Loop* (HITL) que permita a los inspectores corregir falsos positivos desde el panel web, retroalimentando el dataset local para futuros reentrenamientos (*fine-tuning*).
2. **Inyección de datos locales:** Ampliar la recolección pasiva de imágenes específicas de Moreno bajo condiciones variables de iluminación y clima.
3. **Evaluación de RT-DETR:** Investigar y entrenar un detector basado en Transformers en tiempo real (**RT-DETR**). Al utilizar atención global sobre el fotograma completo en lugar de las convoluciones de grilla local de YOLO, se proyecta una mejora sustancial en la detección de anomalías lineales delgadas (grietas `D20`) y objetos pequeños a gran distancia.

## 4. Conclusión

La integración del componente de inteligencia artificial con la infraestructura del sistema representa la culminación del ciclo de vida del proyecto. Al desviar la carga de procesamiento pesado hacia workers distribuidos coordinados de forma asíncrona, se ha logrado consolidar un sistema robusto, modular y apto para ser utilizado en un entorno de producción local.

A modo de síntesis, las conclusiones clave de esta etapa son:

- **De la experimentación a la producción:** Las decisiones analíticas adoptadas (como el escalado a YOLO26 Medium y la resolución de 1024px) demostraron ser determinantes para capturar detalles críticos en calzada. Sin embargo, su efectividad real se consolidó al acoplarlas con las lógicas del backend (como el filtro de horizonte del 50% y los filtros de movimiento).
- **Robustez geoespacial:** El uso de PostGIS y el tracking híbrido (visual con ByteTrack y espacial con `ST_DWithin`) resuelven con éxito la duplicación de daños a lo largo del recorrido. Esto asegura que el modelo no genere alertas repetidas para un mismo desperfecto físico, manteniendo la integridad estadística del inventario vial.
- **Privacidad y modularidad:** La integración local de Ollama (`llama3.2:3b`) permite al municipio generar reportes ejecutivos narrativos e interactivos de manera automática y 100% privada, eliminando la dependencia de APIs cloud costosas y protegiendo los datos geográficos locales.
- **Línea de base funcional:** El sistema integrado constituye una arquitectura de base sólida. Esto facilitará, en etapas posteriores, el reentrenamiento continuo del modelo para corregir limitaciones (como el recall en baches) e incluso la transición hacia otras arquitecturas (como RT-DETR) sin necesidad de modificar el diseño del backend.